In [1]:
from scripts.sglang_inference import SglangModel
from scripts.prompt_processor import PromptProcessor
import torch

In [2]:
processor = PromptProcessor()

In [3]:
print(torch.cuda.is_available())

True


In [6]:
torch.cuda.empty_cache()

# Run Qwen32B model with Docker

In [ ]:
# docker run --gpus all \
#     --shm-size 32g \
#     -p 30000:30000 \
#     -v '/home/ubuntu/models/QwQ-32B':'/models/QwQ-32B' \
#     --ipc=host \
#     lmsysorg/sglang:latest \
#     python3 -m sglang.launch_server \
#     --model-path '/models/QwQ-32B'  \
#     --host 0.0.0.0 \
#     --port 30000 \
#     --enable-p2p-check \
#     --tensor-parallel-size 4 \
#     --dtype bfloat16 \
#     --disable-cuda-graph \
#     --mem-fraction-static 0.9

# Functions for Parsing the Outputs

In [5]:
import re
import json
import pandas as pd
import numpy as np

def extract_first_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    match = re.search(r'\{.*?\}', text)
    if match:
        try:
            json_data = json.loads(match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}

def extract_last_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    # Find all JSON objects in the text
    matches = list(re.finditer(r'\{.*?\}', text))
    if matches:
        # Get the last match
        last_match = matches[-1]
        try:
            json_data = json.loads(last_match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(last_match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}
    
def parse_date_safe(date):
    try:
        return pd.to_datetime(date)
    except:
        return np.nan

# Configuration

In [6]:
CONFIG_QWEN = {
    "model" : {
        "name":"payment_date",
        "generation_params": {
            "batch_size": 1000,
            "max_length" : 7048,
            "max_new_tokens":4000,
            #"max_length" : 2048,
            #"max_new_tokens":1000,
            "temperature": 0.6,
            "do_sample": True,
            "top_p": 0.95,
            "is_completion_model":True
        }
    }
}
PROMT = """You are a document classification system for German after-court / enforcement documents.

TASK
Given the full text of ONE document (often OCR), classify:
1) whether it is a PfüB document (Pfändungs- und Überweisungsbeschluss, order of attachment/garnishment + transfer), and
2) whether an invoice / cost statement / fee payment request is present inside the same document text (either embedded as pages/sections or appended as an “attachment” within the PDF text).

FIELD DEFINITIONS

A) is_pfub = true
Return true if the document is:
- the actual “Pfändungs- und Überweisungsbeschluss” (PfüB / PfÜB), OR
- an official court notice/confirmation stating that an application for a “Pfändungs- und Überweisungsbeschluss” was granted / issued / sent for service / served / delivery initiated (e.g., “dem Antrag … entsprochen”, “Zustellung … veranlasst”), OR
- contains “PfüB” / “PfÜB” and clearly refers to that court order.

Return false if it is not about a PfüB order (e.g., other enforcement letters, reminders, generic correspondence) even if it mentions enforcement in general.

B) is_invoice_inside = true
Return true if the same document text contains a court invoice / cost statement / fee request, including cases where:
- a “Kostenrechnung” or similar is appended as extra pages, OR
- the letter includes a payment request for fees/costs with payment details.

Count as invoice/cost content when there is clear evidence of payment being demanded, such as:
- “Kostenrechnung”
- “Rechnungsbetrag”, “Betrag”, “zu zahlen”, “von Ihnen zu zahlen”
- “Kostenansatz”
- “Kassenzeichen” (or “Kassen-zeichen”)
- fee request wording: “Gebühr”, “Einzahlung”, “bitte zahlen”, “Zahlungsfrist”, “innerhalb von … Wochen/Tagen”
- bank/payment fields: “IBAN”, “BIC”, “Verwendungszweck”, “Zahlungsempfänger”, “Landeshauptkasse”, “Bundeskasse”
- reminders/late fees tied to an invoice: “Mahngebühr”, “Mahnung”, “nach Ablauf der Zahlungsfrist … Einziehung”

Return false if there is no meaningful payment request/cost statement in the text (e.g., only general information, or a stray IBAN in a footer with no invoice/payment request).

ROBUSTNESS / OCR NOISE
Assume OCR errors are common. Treat close variants as matches if clearly intended, e.g.:
- “Überuueisungsbeschluss”, “Übenveisungsbeschluss” ≈ “Überweisungsbeschluss”
- “Kostemrechnung”, “Kostenrechnunq” ≈ “Kostenrechnung”
- spacing / punctuation errors in “PfüB”, “PfÜB”, “IBAN”, etc.

DECISION POLICY
- Be conservative: if uncertain, output false for the uncertain field.
- Use the full text; do not rely on any external knowledge.

OUTPUT RULES (VERY IMPORTANT)
- Output ONLY valid JSON.
- Output exactly two keys: "is_pfub" and "is_invoice_inside"
- Values must be JSON booleans: true or false (lowercase).
- No extra keys, no explanation, no markdown, no surrounding text.

INPUT DOCUMENT TEXT
<<<DOCUMENT_TEXT_START
${clean_text}
DOCUMENT_TEXT_END>>>
"""

In [24]:
data = pd.read_csv("full_egvp_attachment_data_2025-01-20_to_2025-02-27.csv")
data

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,pfub_prob,pfub_slug,pfub_debtor_name,cleaned_text,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at
0,44528753,ocr_source_files/2025-11-21/egvp_id_263308/445...,pair-data-engineering-new,44528753_Dr_ii_788_25_aus_schreibmaschine.pdf,67060082c28ff4555915be5927987865ee1585937169a3...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Gerichtsvollzieher\nAG Pasewalk\nThomas Rex\nZ...,True,NaN,...,NaN,NaN,NaN,gerichtsvollzieher\nag pasewalk\nthomas rex\nz...,gerichtsvollzieher\nag pasewalk\nthomas rex\nz...,0.01,0.120000,4,0,2025-11-21 15:59:17
1,44528763,ocr_source_files/2025-11-21/egvp_id_263309/445...,pair-data-engineering-new,44528763_DR-II_135225_Nachr_GKB-VAKA_21_11_202...,aa0f9ffd827347c6a5092cfc5eaa6f70616717aa9a42f4...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Neue Büroanschrift ab 01.12.2024\nDortmunder S...,False,NaN,...,NaN,NaN,NaN,neue büroanschrift ab 01.12.2024\ndortmunder s...,neue büroanschrift ab <DATE>\ndortmunder str. ...,0.02,0.060000,4,0,2025-11-21 15:59:17
2,44528764,ocr_source_files/2025-11-21/egvp_id_263309/445...,pair-data-engineering-new,44528764_Protokoll_GKB-EV21_19_11_2025__Gl_.pdf,e9aa64c7f95b93fa63717116455de13d0f6984b4b5aaf6...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,RICHARD STIPP\nDie nachstehend gewählten Formu...,True,NaN,...,NaN,NaN,NaN,richard stipp\ndie nachstehend gewählten formu...,richard stipp\ndie nachstehend gewählten formu...,0.00,0.030000,4,0,2025-11-21 15:59:17
3,44528777,ocr_source_files/2025-11-21/egvp_id_263312/445...,pair-data-engineering-new,44528777_DR-II_050625_Nachr_GKB-HBGA_21_11_202...,769f2c7c81621742ac8cc6fbdc1c308187f7c2f6340cb9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,CHRISTOPH JOST\nKohlstr. 55\n66450 Bexbach-Höc...,False,NaN,...,NaN,NaN,NaN,christoph jost\nkohlstr. 55\n66450 bexbach-höc...,christoph jost\nkohlstr. <NUMBER>\n<NUMBER> be...,0.02,0.000000,4,0,2025-11-21 15:59:17
4,44528778,ocr_source_files/2025-11-21/egvp_id_263312/445...,pair-data-engineering-new,44528778_DR-II_050625_Nachr_Uebermittlung_elek...,051c81569a38c7d25956fd1a14e983f44830c9f85c87dd...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,CHRISTOPH JOST\nKohlstr. 55\n66450 Bexbach-Höc...,False,NaN,...,NaN,NaN,NaN,christoph jost\nkohlstr. 55\n66450 bexbach-höc...,christoph jost\nkohlstr. <NUMBER>\n<NUMBER> be...,0.38,0.230000,1,0,2025-11-21 15:59:17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29340,54203493,ocr_source_files/2026-02-27/egvp_id_317018/542...,pair-data-engineering-new,54203493_ANSCHR_Pair_Finance_GmbH_Pair_Finance...,6cc92a56c5f872588c2bc1fd2d5504d130d04decd895cf...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Reutlingen\nVOLLSTRECKUNGSGERICHT\...,False,0.02,...,0.4300,NaN,NaN,amtsgericht reutlingen\nvollstreckungsgericht\...,amtsgericht reutlingen\nvollstreckungsgericht\...,0.04,0.170000,3,0,2026-02-27 06:44:00
29341,54203577,ocr_source_files/2026-02-27/egvp_id_317021/542...,pair-data-engineering-new,54203577_20aM474_26_VB-GL_PairFinanceGmbH_E20E...,674932dadc7a811fdd05b7cbc7299bfa572c483651aa2b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nSyke\nAmtsgericht Syke, Amtshof 2...",False,0.02,...,0.8475,'184509923993','Eve Stummer',"amtsgericht\nsyke\namtsgericht syke, amtshof 2...","amtsgericht\nsyke\namtsgericht syke, amtshof <...",0.00,0.243333,3,0,2026-02-27 06:44:00
29342,54203581,ocr_source_files/2026-02-27/egvp_id_317022/542...,pair-data-engineering-new,54203581_Dokument_185525_27022026_063106.pdf,4b136029ff45376978a64fa58a71dd94e79f2af7bbd483...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Kai Spitzner\nEuroparatstraße 4\nObergerichtsv...,False,0.05,...,0.0000,NaN,NaN,kai spitzner\neuroparatstraße 4\nobergerichtsv..

In [25]:
pfub = data[data['is_pfub']]
pfub
# get 100 not pfub sample and concat with pfub
not_pfub = data[~data['is_pfub']].sample(1000, random_state=42)
filtered_data = pd.concat([pfub, not_pfub], ignore_index=True)
filtered_data.rename(columns={"cleaned_text":"clean_text"}, inplace=True)
filtered_data

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,pfub_prob,pfub_slug,pfub_debtor_name,clean_text,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at
0,50219877,ocr_source_files/2026-01-20/egvp_id_294119/502...,pair-data-engineering-new,50219877_Allgemeines_Schreiben_R__20260120_1f2...,0f7e93fc3bd3b02f7168d8b0b3a829da212839da854b52...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Essen-Borbeck\n-Geschäftsstelle-\n...,False,0.00,...,0.9900,'171607237715','Adomah',amtsgericht essen-borbeck\n-geschäftsstelle-\n...,amtsgericht essen-borbeck\n-geschäftsstelle-\n...,0.00,0.320000,3,0,2026-01-20 10:44:00
1,50219898,ocr_source_files/2026-01-20/egvp_id_294125/502...,pair-data-engineering-new,50219898_Allgemeines_Schreiben_R__20260120_2a2...,26f5d9844c88ad1c35704dc4757acfb52202204049aab6...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Essen\n-30- Amtsgericht Essen, 451...",False,0.00,...,0.9900,'161155294895','Aldag',"amtsgericht essen\n-30- amtsgericht essen, 451...",amtsgericht essen\n<NUMBER>- amtsgericht essen...,0.00,0.348750,3,0,2026-01-20 10:44:00
2,50219909,ocr_source_files/2026-01-20/egvp_id_294128/502...,pair-data-engineering-new,50219909_BriefkopfGericht_20260120_0c3ba229939...,6588a5026d36290c01d535b3a1c0b82ecfc39cd9ff5e7d...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Medebach\n-12- Amtsgericht Medebac...,False,0.00,...,1.0000,'174452615191','Bernau',amtsgericht medebach\n-12- amtsgericht medebac...,amtsgericht medebach\n<NUMBER>- amtsgericht me...,0.00,0.300000,3,0,2026-01-20 10:44:00
3,50219939,ocr_source_files/2026-01-20/egvp_id_294136/502...,pair-data-engineering-new,50219939_4M61_26_GESV-GL_PairFinanceGmbH_70EC2...,3364f178c5530b3a472da9aabe546a60d0055d39d96716...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nWalsrode\nAmtsgericht Walsrode, L...",False,0.01,...,0.7675,'106972047927','Justin Johnson',"amtsgericht\nwalsrode\namtsgericht walsrode, l...","amtsgericht\nwalsrode\namtsgericht walsrode, l...",0.00,0.216333,3,0,2026-01-20 10:44:00
4,50220181,ocr_source_files/2026-01-20/egvp_id_294143/502...,pair-data-engineering-new,50220181_Allgemeines_Schreiben_R__20260120_03c...,5a7282c74de38b4592f081d20417fbfa3a21588c79b214...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Köln\n-286a- Amtsgericht Köln, 509...",False,0.00,...,0.9700,'167989001885','Holtmann',"amtsgericht köln\n-286a- amtsgericht köln, 509...","amtsgericht köln\n-286a- amtsgericht köln, <NU...",0.00,0.348000,3,0,2026-01-20 10:44:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2259,46502572,ocr_source_files/2025-12-11/egvp_id_276782/465...,pair-data-engineering-new,46502572_Dokument_139825_11122025_143428.pdf,1e92a05c47a9080efb2984a38b4bc21a6d6aa5a0e29aa9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Dieter Drong\nHanauer Str. 25, Tel. 06185/8590...",True,NaN,...,NaN,NaN,NaN,"dieter drong\nhanauer str. 25, tel. 06185/8590...","dieter drong\nhanauer str. <NUMBER>, tel. <PHO...",0.13,0.080000,6,0,2025-12-11 14:44:00
2260,48687571,ocr_source_files/2026-01-05/egvp_id_285295/486...,pair-data-engineering-new,48687571_WiderspruchsN_2513814550008E_e43611e7...,0cf6ffca843f52ed4d0557b8a066130347e3476f1f2a54...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Wedding\nMahnsache Liquandum Capit...,False,NaN,...,NaN,NaN,NaN,amtsgericht wedding\nmahnsache liquandum capit...,amtsgericht wedding\nmahnsache liquandum capit...,0.00,0.070000,2,3,2026-01-05 13:44:00
2261,53306537,ocr_source_files/2026-02-18/egvp_id_311964/533...,pair-data-engineering-new,53306537_Dokument_161625_18022026_105814.pdf,c820f0ede2d9fe72349430b730b09fa5f22937516dfe3e...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,K. Hierl\nHeisenbergstraße 4/III\nG

In [ ]:
import time

pd_prompt = {
    "prompt":f"""<|im_start|>user
{PROMT}
<|im_end|>\n<|im_start|>assistant\n<think>\n"""
}
print(f"Running inference with prompt: \n {PROMT}")
texts_val = processor.process(pd_prompt["prompt"], filtered_data.T.to_dict())
texts_val = processor.process(pd_prompt["prompt"], filtered_data.T.to_dict())
print("-"*50)
print(texts_val[0])

Running inference with prompt: 
 You are a document classification system for German after-court / enforcement documents.

TASK
Given the full text of ONE document (often OCR), classify:
1) whether it is a PfüB document (Pfändungs- und Überweisungsbeschluss, order of attachment/garnishment + transfer), and
2) whether an invoice / cost statement / fee payment request is present inside the same document text (either embedded as pages/sections or appended as an “attachment” within the PDF text).

FIELD DEFINITIONS

A) is_pfub = true
Return true if the document is:
- the actual “Pfändungs- und Überweisungsbeschluss” (PfüB / PfÜB), OR
- an official court notice/confirmation stating that an application for a “Pfändungs- und Überweisungsbeschluss” was granted / issued / sent for service / served / delivery initiated (e.g., “dem Antrag … entsprochen”, “Zustellung … veranlasst”), OR
- contains “PfüB” / “PfÜB” and clearly refers to that court order.

Return false if it is not about a PfüB order 

In [29]:
len(texts_val)



2

In [30]:
start = time.time()  # start time
sgl = SglangModel(CONFIG_QWEN)
outs = sgl.predict_batch_complete(texts_val)
end = time.time()
time_spent = end - start
mins = int(time_spent // 60)
secs = int(time_spent % 60)
print(f"{mins} mins {secs} sec")
del sgl
filtered_data[f"pred_llm"] = outs

try:
    filtered_data.to_csv(f"final_preds.csv")
except Exception as e:
    filtered_data.to_pickle(f"final_preds.pkl")
    # save outs
    pd.DataFrame({"pred_llm": outs}).to_csv(f"final_preds_only.csv", index=False)


0 mins 24 sec


In [31]:
filtered_data

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,pfub_slug,pfub_debtor_name,clean_text,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at,pred_llm
603,51882059,ocr_source_files/2026-02-05/egvp_id_304196/518...,pair-data-engineering-new,51882059_4C13_26_PB-KL_RegInkassounternehmenPa...,49ced55b64f21caecb95ddfdaa4bfe2c968a2dc829ede9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nAlfeld\nAmtsgericht Alfeld, Kalan...",False,0.05,...,'193219861781','Bondarchuk',"amtsgericht\nalfeld\namtsgericht alfeld, kalan...","amtsgericht\nalfeld\namtsgericht alfeld, kalan...",0.04,0.140833,3,0,2026-02-05 13:44:00,"Okay, let's tackle this classification task st..."
1868,48863983,ocr_source_files/2026-01-06/egvp_id_285708/488...,pair-data-engineering-new,48863983_Dr_ii_392_25_aus_schreibmaschine.pdf,c8b01c652c2f43d98fa07f72b8e05278d8565f6c322b3f...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin K.Deeben\nDienstkonto...,True,NaN,...,NaN,NaN,obergerichtsvollzieherin k.deeben\ndienstkonto...,obergerichtsvollzieherin k.deeben\ndienstkonto...,0.37,0.190000,1,0,2026-01-06 13:44:00,"Okay, let's tackle this document classificatio..."


In [1]:
import pandas as pd

In [3]:
pred = pd.read_csv("final_preds.csv")

In [4]:
pred

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,pfub_slug,pfub_debtor_name,clean_text,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at,pred_llm
0,50219877,ocr_source_files/2026-01-20/egvp_id_294119/502...,pair-data-engineering-new,50219877_Allgemeines_Schreiben_R__20260120_1f2...,0f7e93fc3bd3b02f7168d8b0b3a829da212839da854b52...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Essen-Borbeck\n-Geschäftsstelle-\n...,False,0.00,...,'171607237715','Adomah',amtsgericht essen-borbeck\n-geschäftsstelle-\n...,amtsgericht essen-borbeck\n-geschäftsstelle-\n...,0.00,0.320000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio..."
1,50219898,ocr_source_files/2026-01-20/egvp_id_294125/502...,pair-data-engineering-new,50219898_Allgemeines_Schreiben_R__20260120_2a2...,26f5d9844c88ad1c35704dc4757acfb52202204049aab6...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Essen\n-30- Amtsgericht Essen, 451...",False,0.00,...,'161155294895','Aldag',"amtsgericht essen\n-30- amtsgericht essen, 451...",amtsgericht essen\n<NUMBER>- amtsgericht essen...,0.00,0.348750,3,0,2026-01-20 10:44:00,"Okay, let's tackle this classification task st..."
2,50219909,ocr_source_files/2026-01-20/egvp_id_294128/502...,pair-data-engineering-new,50219909_BriefkopfGericht_20260120_0c3ba229939...,6588a5026d36290c01d535b3a1c0b82ecfc39cd9ff5e7d...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Medebach\n-12- Amtsgericht Medebac...,False,0.00,...,'174452615191','Bernau',amtsgericht medebach\n-12- amtsgericht medebac...,amtsgericht medebach\n<NUMBER>- amtsgericht me...,0.00,0.300000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio..."
3,50219939,ocr_source_files/2026-01-20/egvp_id_294136/502...,pair-data-engineering-new,50219939_4M61_26_GESV-GL_PairFinanceGmbH_70EC2...,3364f178c5530b3a472da9aabe546a60d0055d39d96716...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nWalsrode\nAmtsgericht Walsrode, L...",False,0.01,...,'106972047927','Justin Johnson',"amtsgericht\nwalsrode\namtsgericht walsrode, l...","amtsgericht\nwalsrode\namtsgericht walsrode, l...",0.00,0.216333,3,0,2026-01-20 10:44:00,"Okay, let's tackle this classification task st..."
4,50220181,ocr_source_files/2026-01-20/egvp_id_294143/502...,pair-data-engineering-new,50220181_Allgemeines_Schreiben_R__20260120_03c...,5a7282c74de38b4592f081d20417fbfa3a21588c79b214...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Köln\n-286a- Amtsgericht Köln, 509...",False,0.00,...,'167989001885','Holtmann',"amtsgericht köln\n-286a- amtsgericht köln, 509...","amtsgericht köln\n-286a- amtsgericht köln, <NU...",0.00,0.348000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2259,46502572,ocr_source_files/2025-12-11/egvp_id_276782/465...,pair-data-engineering-new,46502572_Dokument_139825_11122025_143428.pdf,1e92a05c47a9080efb2984a38b4bc21a6d6aa5a0e29aa9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Dieter Drong\nHanauer Str. 25, Tel. 06185/8590...",True,NaN,...,NaN,NaN,"dieter drong\nhanauer str. 25, tel. 06185/8590...","dieter drong\nhanauer str. <NUMBER>, tel. <PHO...",0.13,0.080000,6,0,2025-12-11 14:44:00,"Okay, let's tackle this classification task. T..."
2260,48687571,ocr_source_files/2026-01-05/egvp_id_285295/486...,pair-data-engineering-new,48687571_WiderspruchsN_2513814550008E_e43611e7...,0cf6ffca843f52ed4d0557b8a066130347e3476f1f2a54...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Wedding\nMahnsache Liquandum Capit...,False,NaN,...,NaN,NaN,amtsgericht wedding\nmahnsache liquandum capit...,amtsgericht wedding\nmahnsache liquandum capit...,0.00,0.070000,2,3,2026-01-05 13:44:00,"Okay, let's

In [6]:
pred["pred_parsed"] = pred["pred_llm"].apply(extract_last_json)

In [7]:
pred["pred_parsed"]

0       {'is_pfub': True, 'is_invoice_inside': False}
1       {'is_pfub': True, 'is_invoice_inside': False}
2       {'is_pfub': True, 'is_invoice_inside': False}
3       {'is_pfub': True, 'is_invoice_inside': False}
4       {'is_pfub': True, 'is_invoice_inside': False}
                            ...                      
2259    {'is_pfub': False, 'is_invoice_inside': True}
2260    {'is_pfub': False, 'is_invoice_inside': True}
2261    {'is_pfub': False, 'is_invoice_inside': True}
2262    {'is_pfub': False, 'is_invoice_inside': True}
2263    {'is_pfub': False, 'is_invoice_inside': True}
Name: pred_parsed, Length: 2264, dtype: object

In [8]:
# take is_pfub and is_invoice_inside from pred_parsed and create new columns for them
pred["pred_is_pfub"] = pred["pred_parsed"].apply(lambda x: x.get("is_pfub", False))
pred["pred_is_invoice_inside"] = pred["pred_parsed"].apply(lambda x: x.get("is_invoice_inside", False))

In [9]:
both_true = pred[(pred["pred_is_pfub"] == True) & (pred["pred_is_invoice_inside"] == True)]
both_true

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at,pred_llm,pred_parsed,pred_is_pfub,pred_is_invoice_inside
1138,53789376,ocr_source_files/2026-02-24/egvp_id_314455/537...,pair-data-engineering-new,53789376_718M186461_25_VB-GL_PairFinanceGmbH_B...,234b2a566ff2ca1b76819aebf5efbd4826ec3738a3eb44...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nHannover\nAmtsgericht Hannover, V...",False,0.03,...,"amtsgericht\nhannover\namtsgericht hannover, v...",0.01,0.193333,3,0,2026-02-24 08:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1378,53417764,ocr_source_files/2026-02-20/egvp_id_313477/534...,pair-data-engineering-new,53417764_Dokument_107026_20022026_165432.pdf,a7b6cb94d8e8b56a761c6bc0fd0f0495a437f12601bd0d...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Carina Frei\nPostfach 11 03\nObergerichtsvollz...,False,0.02,...,carina frei\npostfach <NUMBER> <NUMBER>\noberg...,0.04,0.140000,6,0,2026-02-20 20:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1415,50426388,ocr_source_files/2026-01-21/egvp_id_295065/504...,pair-data-engineering-new,50426388_DR_83-26_ZU.pdf,1183d6219297c55cc62570d05c3d64dfb83a764c7f90ac...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,FRANK SCHROER\nDie Zustellung konnte aus folge...,False,0.17,...,frank schroer\ndie zustellung konnte aus folge...,0.05,0.100000,4,0,2026-01-21 15:44:00,"Okay, let's tackle this classification task st...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1480,46499574,ocr_source_files/2025-12-11/egvp_id_276711/464...,pair-data-engineering-new,46499574_Dokument_210325_11122025_124445.pdf,76c2e7de85a4e46c3542ac861acad5e74bcea74aafe20a...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Obergerichtsvollzieherin Julia Gerlach\nPostfa...,False,NaN,...,obergerichtsvollzieherin julia gerlach\npostfa...,0.00,0.140000,6,0,2025-12-11 13:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1499,49803011,ocr_source_files/2026-01-15/egvp_id_291727/498...,pair-data-engineering-new,49803011_Dokument_6426_15012026_114841.pdf,31ea6441d383f26ad38175eaf61fa4ee354e8bc6ecde69...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,A. Beier\nErnst-Reuter-Straße 20 B\nObergerich...,True,NaN,...,a. beier\nernst-reuter-straße <NUMBER> b\nober...,0.14,0.110000,6,0,2026-01-15 12:44:00,"Okay, let's tackle this classification step by...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1547,49775445,ocr_source_files/2026-01-14/egvp_id_291251/497...,pair-data-engineering-new,49775445_0_sammel3.pdf,9ee74cd9e7336469e2b73a0bc672f09c3238869fc2cfdc...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Bitte stets angeben:\nDR 1214/25\nDer Auftrag ...,True,NaN,...,bitte stets angeben:\ndr <NUMBER>/<NUMBER>\nde...,0.23,0.000000,1,0,2026-01-14 15:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1627,45358291,ocr_source_files/2025-11-29/egvp_id_267995/453...,pair-data-engineering-new,45358291_Dokument_152725_29112025_101139.pdf,a5b72b5d8992f76f5d07d0a5f3f53c6de7af3cf5c37dca...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,M. Ibsch\nBrunsbütteler Damm 446\nGerichtsvoll...,False,NaN,...,m. ibsch\nbrunsbütteler damm <NUMBER>\ngericht...,0.01,0.110000,6,0,2025-11-29 11:01:14,"Okay, let's tackle this classification task st...","{'is_pfub': True, 'is_invoice_inside': True}",True,True
1646,50762520,ocr_source_files/2026-01-26/egvp_id_297010/507...,pair-data-engineering-new,50762520_DR-I_002626_Nachr_Ruecksendung_Titel_...,1b9d64091c1cbe37f1cf0f72c7a42b7325f1f0ef64ee12...,SUCCEEDED,s3://pair

In [10]:
import boto3
import sys
sys.path.append("../../")
from utils.prod_utils import download_pdf_from_s3

s3 = boto3.client("s3")

download_dir = "pfub_invoices"

for _, row in both_true.iterrows():
    download_pdf_from_s3(
        s3=s3,
        bucket_name=row["document_s3_bucket"],
        object_key=row["document_s3_key"],
        download_dir=download_dir,
        file_name=row.get("file_name", None)
    )

✅ PDF downloaded successfully: pfub_invoices/53789376_718M186461_25_VB-GL_PairFinanceGmbH_BC368E84-BDD0-4A27-BA1A-0B9998E4D520.pdf
✅ PDF downloaded successfully: pfub_invoices/53417764_Dokument_107026_20022026_165432.pdf
✅ PDF downloaded successfully: pfub_invoices/50426388_DR_83-26_ZU.pdf
✅ PDF downloaded successfully: pfub_invoices/46499574_Dokument_210325_11122025_124445.pdf
✅ PDF downloaded successfully: pfub_invoices/49803011_Dokument_6426_15012026_114841.pdf
✅ PDF downloaded successfully: pfub_invoices/49775445_0_sammel3.pdf
✅ PDF downloaded successfully: pfub_invoices/45358291_Dokument_152725_29112025_101139.pdf
✅ PDF downloaded successfully: pfub_invoices/50762520_DR-I_002626_Nachr_Ruecksendung_Titel_26_01_2026.pdf
✅ PDF downloaded successfully: pfub_invoices/47983841_DR-II_169925_Nachr_Textbaustein_pzur_nicht__mehr__vorhanden__28_12_2025.pdf
✅ PDF downloaded successfully: pfub_invoices/48862727_Dokument_243725_06012026_101815.pdf
✅ PDF downloaded successfully: pfub_invoices/45

In [11]:
pred

,attachment_id,document_s3_key,document_s3_bucket,file_name,textract_job_id,textract_status,textract_s3_link,text,is_ladung,ladung_prob,...,text_with_tags,ladung_prob_cur,pfub_prob_cur,kmeans_cluster,dbscan_cluster,created_at,pred_llm,pred_parsed,pred_is_pfub,pred_is_invoice_inside
0,50219877,ocr_source_files/2026-01-20/egvp_id_294119/502...,pair-data-engineering-new,50219877_Allgemeines_Schreiben_R__20260120_1f2...,0f7e93fc3bd3b02f7168d8b0b3a829da212839da854b52...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Essen-Borbeck\n-Geschäftsstelle-\n...,False,0.00,...,amtsgericht essen-borbeck\n-geschäftsstelle-\n...,0.00,0.320000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': False}",True,False
1,50219898,ocr_source_files/2026-01-20/egvp_id_294125/502...,pair-data-engineering-new,50219898_Allgemeines_Schreiben_R__20260120_2a2...,26f5d9844c88ad1c35704dc4757acfb52202204049aab6...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Essen\n-30- Amtsgericht Essen, 451...",False,0.00,...,amtsgericht essen\n<NUMBER>- amtsgericht essen...,0.00,0.348750,3,0,2026-01-20 10:44:00,"Okay, let's tackle this classification task st...","{'is_pfub': True, 'is_invoice_inside': False}",True,False
2,50219909,ocr_source_files/2026-01-20/egvp_id_294128/502...,pair-data-engineering-new,50219909_BriefkopfGericht_20260120_0c3ba229939...,6588a5026d36290c01d535b3a1c0b82ecfc39cd9ff5e7d...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Medebach\n-12- Amtsgericht Medebac...,False,0.00,...,amtsgericht medebach\n<NUMBER>- amtsgericht me...,0.00,0.300000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': False}",True,False
3,50219939,ocr_source_files/2026-01-20/egvp_id_294136/502...,pair-data-engineering-new,50219939_4M61_26_GESV-GL_PairFinanceGmbH_70EC2...,3364f178c5530b3a472da9aabe546a60d0055d39d96716...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht\nWalsrode\nAmtsgericht Walsrode, L...",False,0.01,...,"amtsgericht\nwalsrode\namtsgericht walsrode, l...",0.00,0.216333,3,0,2026-01-20 10:44:00,"Okay, let's tackle this classification task st...","{'is_pfub': True, 'is_invoice_inside': False}",True,False
4,50220181,ocr_source_files/2026-01-20/egvp_id_294143/502...,pair-data-engineering-new,50220181_Allgemeines_Schreiben_R__20260120_03c...,5a7282c74de38b4592f081d20417fbfa3a21588c79b214...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Amtsgericht Köln\n-286a- Amtsgericht Köln, 509...",False,0.00,...,"amtsgericht köln\n-286a- amtsgericht köln, <NU...",0.00,0.348000,3,0,2026-01-20 10:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': True, 'is_invoice_inside': False}",True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2259,46502572,ocr_source_files/2025-12-11/egvp_id_276782/465...,pair-data-engineering-new,46502572_Dokument_139825_11122025_143428.pdf,1e92a05c47a9080efb2984a38b4bc21a6d6aa5a0e29aa9...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,"Dieter Drong\nHanauer Str. 25, Tel. 06185/8590...",True,NaN,...,"dieter drong\nhanauer str. <NUMBER>, tel. <PHO...",0.13,0.080000,6,0,2025-12-11 14:44:00,"Okay, let's tackle this classification task. T...","{'is_pfub': False, 'is_invoice_inside': True}",False,True
2260,48687571,ocr_source_files/2026-01-05/egvp_id_285295/486...,pair-data-engineering-new,48687571_WiderspruchsN_2513814550008E_e43611e7...,0cf6ffca843f52ed4d0557b8a066130347e3476f1f2a54...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,Amtsgericht Wedding\nMahnsache Liquandum Capit...,False,NaN,...,amtsgericht wedding\nmahnsache liquandum capit...,0.00,0.070000,2,3,2026-01-05 13:44:00,"Okay, let's tackle this document classificatio...","{'is_pfub': False, 'is_invoice_inside': True}",False,True
2261,53306537,ocr_source_files/2026-02-1

In [13]:
pred.is_pfub.value_counts

<bound method IndexOpsMixin.value_counts of 0        True
1        True
2        True
3        True
4        True
        ...  
2259    False
2260    False
2261    False
2262    False
2263    False
Name: is_pfub, Length: 2264, dtype: bool>